In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

In [47]:
df = pd.read_csv('/content/drive/MyDrive/EG/translated_dataset/mbart50/mbart50_p6.csv')

In [48]:
df = df.reset_index(drop=False)

Add a column with proper indicing if not present

In [49]:
df

,index,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m
0,0,8886,14834,6,There were many obstacles that the builders fa...,2,एम्पायर स्टेट बिल्डिंग में डिरिगिबल्स डॉक करने...,इम्पर स्टेट भवनमा dirigibles डक गर्ने प्रयास ग...
1,1,8887,14835,6,"Him from the start, there would have been many...",3,"उसे शुरू से ही, हवा में कुछ @ NUM1 फीट डॉक करन...","उहाँलाई सुरुदेखि नै, केही @NUM1 फीट आकाशमा डक ..."
2,2,8888,14836,6,The builders of the Empire State Building face...,4,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां डॉक...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...
3,3,8889,14837,6,In the passage The Mooring Mast by Marcia Amid...,1,मार्ग में मूरिंग मस्त मार्सिया एमिडॉन @ CAPS1 ...,मार्गमा मार्सिया अमीडन @CAPS1 द्वारा निर्माण ग...
4,4,8890,14838,6,The builders of the Empire State Building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को डिरिगिबल...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...
...,...,...,...,...,...,...,...,...
1795,1795,10681,16629,6,The one obstacle the builders had when trying ...,0,इस इमारत को बनाने की कोशिश करते समय बिल्डरों क...,निर्माणकर्ताहरूले यो भवन निर्माण गर्ने प्रयास ...
1796,1796,10682,16630,6,Some of the problems with the constructing of ...,2,डॉकिंग डिरिगिबल्स के निर्माण के साथ कुछ समस्या...,डकिङ डाइरिभेलको निर्माणमा केही समस्याहरू निम्न...
1797,1797,10683,16631,6,The builders of the Empire State building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां कुछ...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...
1798,1798,10684,16632,6,The obstacles the builders of the Empire State...,2,एम्पायर स्टेट बिल्डिंग के बिल्डरों की बाधाएं य...,इम्पर स्टेट भवनका निर्माणकर्ताहरूका अवरोधहरू य...


In [50]:
def addCol(x):
  return x

Preprocessing

In [51]:
import re
def remove_mentions(x):
  if x=='nan':
    return ""
  x = re.sub("@[a-zA-Z0-9 ]+", " ", x)
  return x

In [52]:
df.mbart50_m2m = df.mbart50_m2m.apply( lambda x: remove_mentions(str(x)))

In [53]:
df.head()

,index,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m
0,0,8886,14834,6,There were many obstacles that the builders fa...,2,एम्पायर स्टेट बिल्डिंग में डिरिगिबल्स डॉक करने...,इम्पर स्टेट भवनमा dirigibles डक गर्ने प्रयास ग...
1,1,8887,14835,6,"Him from the start, there would have been many...",3,"उसे शुरू से ही, हवा में कुछ @ NUM1 फीट डॉक करन...","उहाँलाई सुरुदेखि नै, केही फीट आकाशमा डक गर्न ..."
2,2,8888,14836,6,The builders of the Empire State Building face...,4,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां डॉक...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...
3,3,8889,14837,6,In the passage The Mooring Mast by Marcia Amid...,1,मार्ग में मूरिंग मस्त मार्सिया एमिडॉन @ CAPS1 ...,मार्गमा मार्सिया अमीडन द्वारा निर्माण गरिएको ...
4,4,8890,14838,6,The builders of the Empire State Building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को डिरिगिबल...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...


In [54]:
#stopwords removal
stop_words = []
with open('/content/drive/MyDrive/EG/nepali_stopwords.txt') as f:
    for line in f:
        stop_words.append(line.strip())

In [55]:
def vocab_size(x):
  #removing punctuation
  x = remove_mentions(x)

  x = x.replace("।", " ")
  x = x.replace("!", " ")
  x = x.replace("@", " ")
  x = x.replace(",", " ")
  x = x.replace("?", " ")

  #extracting words
  vocab = x.split()

  final_vocab = []

  for word in vocab:
    if word != " " and word not in stop_words:
        final_vocab.append(word)

  dict = {}

  for word in final_vocab:
    if word not in dict:
      dict.update({ word : 1 })
    else:
      dict[word] += 1

  cnt = 0

  for key, value in dict.items():
      cnt = cnt + 1

  return cnt

In [56]:
df['vocab_size'] = df.mbart50_m2m.apply( lambda x: vocab_size(x))

In [57]:
def rare_words(x):
  #removing punctuation
  x = x.replace("।", " ")
  x = x.replace("!", " ")
  x = x.replace("@", " ")
  x = x.replace(",", " ")
  x = x.replace("?", " ")

  #extracting words
  vocab = x.split()

  final_vocab = []

  for word in vocab:
    if word != " " and word not in stop_words:
        final_vocab.append(word)

  dict = {}

  for word in final_vocab:
    if word not in dict:
      dict.update({ word : 1 })
    else:
      dict[word] += 1

  cnt = 0
  for key, value in dict.items():
    if value == 1:
      cnt = cnt + 1

  return cnt

In [58]:
df['unique_words_count'] = df.mbart50_m2m.apply( lambda x: rare_words(x))

In [59]:
df.head()

,index,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m,vocab_size,unique_words_count
0,0,8886,14834,6,There were many obstacles that the builders fa...,2,एम्पायर स्टेट बिल्डिंग में डिरिगिबल्स डॉक करने...,इम्पर स्टेट भवनमा dirigibles डक गर्ने प्रयास ग...,57,50
1,1,8887,14835,6,"Him from the start, there would have been many...",3,"उसे शुरू से ही, हवा में कुछ @ NUM1 फीट डॉक करन...","उहाँलाई सुरुदेखि नै, केही फीट आकाशमा डक गर्न ...",78,69
2,2,8888,14836,6,The builders of the Empire State Building face...,4,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां डॉक...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,73,62
3,3,8889,14837,6,In the passage The Mooring Mast by Marcia Amid...,1,मार्ग में मूरिंग मस्त मार्सिया एमिडॉन @ CAPS1 ...,मार्गमा मार्सिया अमीडन द्वारा निर्माण गरिएको ...,101,90
4,4,8890,14838,6,The builders of the Empire State Building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को डिरिगिबल...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,72,60


**Semantic Overlap**

In [60]:
!pip install tensorflow_text

In [61]:
!pip install bert-for-tf2

In [62]:
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_hub as hub
import tensorflow_text as text
from bert import bert_tokenization
import numpy as np
from scipy.spatial import distance

def get_model(model_url, max_seq_length):
  inputs = dict(
    input_word_ids=tf.keras.layers.Input(shape=(max_seq_length,), dtype=tf.int32),
    input_mask=tf.keras.layers.Input(shape=(max_seq_length,), dtype=tf.int32),
    input_type_ids=tf.keras.layers.Input(shape=(max_seq_length,), dtype=tf.int32),
    )
  muril_layer = hub.KerasLayer(model_url, trainable=True)
  outputs = muril_layer(inputs)
  assert 'sequence_output' in outputs
  assert 'pooled_output' in outputs
  assert 'encoder_outputs' in outputs
  assert 'default' in outputs
  return tf.keras.Model(inputs=inputs,outputs=outputs["pooled_output"]), muril_layer
#function call
max_seq_length = 128
muril_model, muril_layer = get_model(
    model_url="https://tfhub.dev/google/MuRIL/1", max_seq_length=max_seq_length)

vocab_file = muril_layer.resolved_object.vocab_file.asset_path.numpy()
do_lower_case = muril_layer.resolved_object.do_lower_case.numpy()
tokenizer = bert_tokenization.FullTokenizer(vocab_file, do_lower_case)

def create_input(input_strings, tokenizer, max_seq_length):
  input_ids_all, input_mask_all, input_type_ids_all = [], [], []
  for input_string in input_strings:
    input_tokens = ["[CLS]"] + tokenizer.tokenize(input_string) + ["[SEP]"]
    input_ids = tokenizer.convert_tokens_to_ids(input_tokens)
    sequence_length = min(len(input_ids), max_seq_length)
    if len(input_ids) >= max_seq_length:
      input_ids = input_ids[:max_seq_length]
    else:
      input_ids = input_ids + [0] * (max_seq_length - len(input_ids))
    input_mask = [1] * sequence_length + [0] * (max_seq_length - sequence_length)
    input_ids_all.append(input_ids)
    input_mask_all.append(input_mask)
    input_type_ids_all.append([0] * max_seq_length)
  return np.array(input_ids_all), np.array(input_mask_all), np.array(input_type_ids_all)

def encode(input_text):
  input_ids, input_mask, input_type_ids = create_input(input_text,tokenizer,max_seq_length)
  inputs = dict(
      input_word_ids=input_ids,
      input_mask=input_mask,
      input_type_ids=input_type_ids,
  )
  return muril_model(inputs)

In [63]:
def get_overlap(x):
  sentences = x.split("।")
  scores = []

  n = len(sentences)
  prev = 0
  curr = 4

  while curr < n:
    temp = []
    temp.append(sentences[prev])
    temp.append(sentences[curr])

    embeddings = encode(temp)

    scores.append(distance.cosine(np.array(embeddings[0]), np.array(embeddings[1])))

    prev = prev + 4
    curr = curr + 4

  curr = curr%n
  temp = []
  temp.append(sentences[prev])
  temp.append(sentences[curr])

  embeddings = encode(temp)

  scores.append(distance.cosine(np.array(embeddings[0]), np.array(embeddings[1])))


  #returning avg semantic overlap score
  return sum(scores)/len(scores)

In [64]:
df['overlap_score'] = df.mbart50_m2m.apply(lambda x: get_overlap(x))

In [65]:
df.head()

,index,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m,vocab_size,unique_words_count,overlap_score
0,0,8886,14834,6,There were many obstacles that the builders fa...,2,एम्पायर स्टेट बिल्डिंग में डिरिगिबल्स डॉक करने...,इम्पर स्टेट भवनमा dirigibles डक गर्ने प्रयास ग...,57,50,0.001299
1,1,8887,14835,6,"Him from the start, there would have been many...",3,"उसे शुरू से ही, हवा में कुछ @ NUM1 फीट डॉक करन...","उहाँलाई सुरुदेखि नै, केही फीट आकाशमा डक गर्न ...",78,69,0.001432
2,2,8888,14836,6,The builders of the Empire State Building face...,4,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां डॉक...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,73,62,0.000842
3,3,8889,14837,6,In the passage The Mooring Mast by Marcia Amid...,1,मार्ग में मूरिंग मस्त मार्सिया एमिडॉन @ CAPS1 ...,मार्गमा मार्सिया अमीडन द्वारा निर्माण गरिएको ...,101,90,0.001175
4,4,8890,14838,6,The builders of the Empire State Building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को डिरिगिबल...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,72,60,0.001339


In [66]:
import re
re.findall("@[a-zA-Z0-9 ]+", df.mbart50_m2m[0])

[]

**Statistical Features** - Essay Length, Average Sentence Length, Average Word Length and Readability Score

In [67]:
# df['Unnamed: 0'] = np.arange(len(df))
# # to add indexing

In [68]:
df

,index,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m,vocab_size,unique_words_count,overlap_score
0,0,8886,14834,6,There were many obstacles that the builders fa...,2,एम्पायर स्टेट बिल्डिंग में डिरिगिबल्स डॉक करने...,इम्पर स्टेट भवनमा dirigibles डक गर्ने प्रयास ग...,57,50,0.001299
1,1,8887,14835,6,"Him from the start, there would have been many...",3,"उसे शुरू से ही, हवा में कुछ @ NUM1 फीट डॉक करन...","उहाँलाई सुरुदेखि नै, केही फीट आकाशमा डक गर्न ...",78,69,0.001432
2,2,8888,14836,6,The builders of the Empire State Building face...,4,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां डॉक...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,73,62,0.000842
3,3,8889,14837,6,In the passage The Mooring Mast by Marcia Amid...,1,मार्ग में मूरिंग मस्त मार्सिया एमिडॉन @ CAPS1 ...,मार्गमा मार्सिया अमीडन द्वारा निर्माण गरिएको ...,101,90,0.001175
4,4,8890,14838,6,The builders of the Empire State Building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को डिरिगिबल...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,72,60,0.001339
...,...,...,...,...,...,...,...,...,...,...,...
1795,1795,10681,16629,6,The one obstacle the builders had when trying ...,0,इस इमारत को बनाने की कोशिश करते समय बिल्डरों क...,निर्माणकर्ताहरूले यो भवन निर्माण गर्ने प्रयास ...,54,45,0.000524
1796,1796,10682,16630,6,Some of the problems with the constructing of ...,2,डॉकिंग डिरिगिबल्स के निर्माण के साथ कुछ समस्या...,डकिङ डाइरिभेलको निर्माणमा केही समस्याहरू निम्न...,33,32,0.000000
1797,1797,10683,16631,6,The builders of the Empire State building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां कुछ...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,53,44,0.000000
1798,1798,10684,16632,6,The obstacles the builders of the Empire State...,2,एम्पायर स्टेट बिल्डिंग के बिल्डरों की बाधाएं य...,इम्पर स्टेट भवनका निर्माणकर्ताहरूका अवरोधहरू य...,30,24,0.000000


In [69]:
!unzip /content/drive/MyDrive/EG/model/indic_nlp_library-master.zip

Archive:  /content/drive/MyDrive/EG/model/indic_nlp_library-master.zip
05bb7c93b63d32729f576028b5ace0efc2396138
replace indic_nlp_library-master/LICENSE? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [70]:
!unzip /content/drive/MyDrive/EG/model/indic_nlp_resources.zip

Archive:  /content/drive/MyDrive/EG/model/indic_nlp_resources.zip
replace indic_nlp_resources/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [71]:
!pip install indic-nlp-library

In [72]:
import sys
from indicnlp import common
from indicnlp import loader
# The path to the local git repo for Indic NLP library
INDIC_NLP_LIB_HOME=r"/content/indic_nlp_library-master"

# The path to the local git repo for Indic NLP Resources
INDIC_NLP_RESOURCES=r"/content/indic_nlp_resources"

sys.path.append(r'{}'.format(INDIC_NLP_LIB_HOME))

common.set_resources_path(INDIC_NLP_RESOURCES)
loader.load()

In [73]:
from indicnlp.syllable import  syllabifier
lang='ne'

In [74]:
# data = [['goo','प्रिय स्थानीय अखबार मलाई']]
# df = pd.DataFrame(data,columns=['xyz','google'])

In [75]:
# df

In [76]:
essays=df["mbart50_m2m"]
essays

0       इम्पर स्टेट भवनमा dirigibles डक गर्ने प्रयास ग...
1       उहाँलाई सुरुदेखि नै, केही  फीट आकाशमा डक गर्न ...
2       इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...
3       मार्गमा मार्सिया अमीडन  द्वारा निर्माण गरिएको ...
4       इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...
                              ...                        
1795    निर्माणकर्ताहरूले यो भवन निर्माण गर्ने प्रयास ...
1796    डकिङ डाइरिभेलको निर्माणमा केही समस्याहरू निम्न...
1797    इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...
1798    इम्पर स्टेट भवनका निर्माणकर्ताहरूका अवरोधहरू य...
1799    तपाईँले मलाई बताउन चाहनुहुन्छ तिनीहरूलाई त्यहा...
Name: mbart50_m2m, Length: 1800, dtype: object

In [77]:
l_of_features = []
readability_score = []

for i in range(len(essays)):
    no_of_words = 0
    no_of_sentences = 0
    total_len_of_words = 0
    no_poly_syllables = 0
    essay = essays[i]

    if essay == "":
        l_of_features.append([0, 0, 0])
        readability_score.append(0)
        continue

    # One essay
    essay = essay.replace("!", "।")
    essay = essay.replace("?", "।")
    temp = essay.split("।")

    # Number of sentences
    no_of_sentences = len(temp)

    for j in range(len(temp)):
        b = temp[j].replace(",", " ").split()
        no_of_words += len(b)

        for k in range(len(b)):
            total_len_of_words += len(b[k])

            # Assumed strictly greater than 2
            if len(' '.join(syllabifier.orthographic_syllabify_improved(b[k], lang))) > 2:
                no_poly_syllables += 1

    # Check for division by zero
    if no_of_sentences == 0 or no_of_words == 0:
        l_of_features.append([no_of_words, 0, 0])
    else:
        # Check for division by zero
        if no_of_words == 0:
            l_of_features.append([no_of_words, 0, 0])
        else:
            l_of_features.append([no_of_words, (no_of_words / no_of_sentences), (total_len_of_words / no_of_words)])

    temp5 = -2.34 + (2.14 * (total_len_of_words / (no_of_words + 1))) + (0.01 * no_poly_syllables)
    readability_score.append(temp5)

In [78]:
# l_of_features=[]
# #essay_length, avg_sentence_length, avg_word_length
# readability_score=[]
# for i in range(len(essays)):
#   no_of_words=0
#   no_of_sentences=0
#   total_len_of_words=0
#   no_poly_syllables=0
#   essay=essays[i]
#   if essay=="":
#     l_of_features.append([0,0,0])
#     readability_score.append(0)
#     continue
#   #one essay
#   essay = essay.replace("!","।")
#   essay = essay.replace("?","।")
#   temp=essay.split("।")
#   # print(temp)
#   #print(temp) number of sentences
#   no_of_sentences=len(temp)
#   for j in range(len(temp)):
#     b=temp[j].replace(","," ").split()
#     no_of_words+=len(b)
#     for k in range(len(b)):
#       total_len_of_words+=len(b[k])
#       #assumed strictly greater than 2
#       if(len(' '.join(syllabifier.orthographic_syllabify_improved(b[k],lang)).split()))>2:
#         no_poly_syllables+=1
#         # print(no_poly_syllables)
#   l_of_features.append([no_of_words,(no_of_words/no_of_sentences),(total_len_of_words/no_of_words)])
#   temp5=-2.34+(2.14*(total_len_of_words/no_of_words))+(0.01*no_poly_syllables)
#   readability_score.append(temp5)

In [79]:
l_of_features

[[92, 13.142857142857142, 5.934782608695652],
 [138, 13.8, 5.427536231884058],
 [117, 13.0, 5.769230769230769],
 [157, 15.7, 5.484076433121019],
 [128, 11.636363636363637, 5.65625],
 [129, 11.727272727272727, 5.666666666666667],
 [106, 15.142857142857142, 5.745283018867925],
 [103, 14.714285714285714, 5.533980582524272],
 [100, 10.0, 5.62],
 [103, 7.923076923076923, 5.533980582524272],
 [190, 11.875, 5.152631578947369],
 [235, 12.368421052631579, 5.940425531914894],
 [150, 13.636363636363637, 5.253333333333333],
 [155, 11.923076923076923, 5.548387096774194],
 [97, 16.166666666666668, 5.536082474226804],
 [125, 15.625, 5.6],
 [111, 13.875, 5.324324324324325],
 [96, 12.0, 5.510416666666667],
 [72, 18.0, 5.013888888888889],
 [59, 11.8, 5.084745762711864],
 [88, 8.8, 5.75],
 [181, 15.083333333333334, 5.033149171270718],
 [134, 10.307692307692308, 5.559701492537314],
 [178, 12.714285714285714, 5.230337078651686],
 [96, 16.0, 5.25],
 [128, 11.636363636363637, 5.3984375],
 [148, 14.8, 5.44594

In [80]:
def essay_length(x):
  # no_of_words=0
  # no_of_sentences=0
  # total_len_of_words=0
  # no_poly_syllables=0
  # essay=x
  # #one essay
  # essay = essay.replace("!","।")
  # essay = essay.replace("?","।")
  # temp=essay.split("।")
  # # print(temp)
  # #print(temp) number of sentences
  # no_of_sentences=len(temp)
  # for j in range(len(temp)):
  #   b=temp[j].replace(","," ").split()
  #   no_of_words+=len(b)
  # return no_of_words
  return l_of_features[x][0]


In [81]:
df['essay_length'] = df["index"].apply(lambda x:essay_length(x))

In [82]:
df

,index,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m,vocab_size,unique_words_count,overlap_score,essay_length
0,0,8886,14834,6,There were many obstacles that the builders fa...,2,एम्पायर स्टेट बिल्डिंग में डिरिगिबल्स डॉक करने...,इम्पर स्टेट भवनमा dirigibles डक गर्ने प्रयास ग...,57,50,0.001299,92
1,1,8887,14835,6,"Him from the start, there would have been many...",3,"उसे शुरू से ही, हवा में कुछ @ NUM1 फीट डॉक करन...","उहाँलाई सुरुदेखि नै, केही फीट आकाशमा डक गर्न ...",78,69,0.001432,138
2,2,8888,14836,6,The builders of the Empire State Building face...,4,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां डॉक...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,73,62,0.000842,117
3,3,8889,14837,6,In the passage The Mooring Mast by Marcia Amid...,1,मार्ग में मूरिंग मस्त मार्सिया एमिडॉन @ CAPS1 ...,मार्गमा मार्सिया अमीडन द्वारा निर्माण गरिएको ...,101,90,0.001175,157
4,4,8890,14838,6,The builders of the Empire State Building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को डिरिगिबल...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,72,60,0.001339,128
...,...,...,...,...,...,...,...,...,...,...,...,...
1795,1795,10681,16629,6,The one obstacle the builders had when trying ...,0,इस इमारत को बनाने की कोशिश करते समय बिल्डरों क...,निर्माणकर्ताहरूले यो भवन निर्माण गर्ने प्रयास ...,54,45,0.000524,113
1796,1796,10682,16630,6,Some of the problems with the constructing of ...,2,डॉकिंग डिरिगिबल्स के निर्माण के साथ कुछ समस्या...,डकिङ डाइरिभेलको निर्माणमा केही समस्याहरू निम्न...,33,32,0.000000,52
1797,1797,10683,16631,6,The builders of the Empire State building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां कुछ...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,53,44,0.000000,83
1798,1798,10684,16632,6,The obstacles the builders of the Empire State...,2,एम्पायर स्टेट बिल्डिंग के बिल्डरों की बाधाएं य...,इम्पर स्टेट भवनका निर्माणकर्ताहरूका अवरोधहरू य...,30,24,0.000000,54


In [83]:
def avg_sentence_length(x):
  return l_of_features[x][1]

In [84]:
df['average_sentence_length']=df["index"].apply(lambda x:avg_sentence_length(x))

In [85]:
def avg_word_length(x):
  return l_of_features[x][2]

In [86]:
df['average_word_length']=df["index"].apply(lambda x:avg_word_length(x))

In [87]:
def readability(x):
  return readability_score[x]

In [88]:
df['readability']=df["index"].apply(lambda x:readability(x))

In [89]:
df.head()

,index,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m,vocab_size,unique_words_count,overlap_score,essay_length,average_sentence_length,average_word_length,readability
0,0,8886,14834,6,There were many obstacles that the builders fa...,2,एम्पायर स्टेट बिल्डिंग में डिरिगिबल्स डॉक करने...,इम्पर स्टेट भवनमा dirigibles डक गर्ने प्रयास ग...,57,50,0.001299,92,13.142857,5.934783,11.093871
1,1,8887,14835,6,"Him from the start, there would have been many...",3,"उसे शुरू से ही, हवा में कुछ @ NUM1 फीट डॉक करन...","उहाँलाई सुरुदेखि नै, केही फीट आकाशमा डक गर्न ...",78,69,0.001432,138,13.800000,5.427536,10.471367
2,2,8888,14836,6,The builders of the Empire State Building face...,4,एम्पायर स्टेट बिल्डिंग के बिल्डरों को वहां डॉक...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,73,62,0.000842,117,13.000000,5.769231,11.031525
3,3,8889,14837,6,In the passage The Mooring Mast by Marcia Amid...,1,मार्ग में मूरिंग मस्त मार्सिया एमिडॉन @ CAPS1 ...,मार्गमा मार्सिया अमीडन द्वारा निर्माण गरिएको ...,101,90,0.001175,157,15.700000,5.484076,10.771646
4,4,8890,14838,6,The builders of the Empire State Building face...,3,एम्पायर स्टेट बिल्डिंग के बिल्डरों को डिरिगिबल...,इम्पर स्टेट भवनका निर्माणकर्ताहरूले जहाजलाई त्...,72,60,0.001339,128,11.636364,5.656250,10.950543


In [90]:
df.to_csv('/content/drive/MyDrive/EG/translated_dataset/mbart50/mbart50_p6_fe.csv', index=False)